In [3]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [8]:
!pip install flask pyngrok pydub librosa soundfile tensorflow requests


In [5]:
!apt-get install ffmpeg

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 41 not upgraded.


In [16]:
import os
import numpy as np
import librosa
from pydub import AudioSegment
from flask import Flask, request, jsonify
import tensorflow as tf
from pyngrok import ngrok
import threading

# ----------------------------
# 1) NGROK TOKEN
# ----------------------------
ngrok.set_auth_token("3620toMGts6dF9yvlucrWEVRV7l_3WWRPkAhRYdfmhHzv1mE4")

# ----------------------------
# 2) LOAD MODEL
# ----------------------------
MODEL_PATH = "/content/drive/MyDrive/models/voice_model_final.h5"
model = tf.keras.models.load_model(MODEL_PATH)

labels = ["ch3al", "tfi", "sini bzarba", "sini bchwiya"]

# ----------------------------
# 3) FEATURE EXTRACTION
# ----------------------------
def extract_features(path, max_len=118):
    y, sr = librosa.load(path, sr=22050)

    # Extract MFCCs: shape (40, time)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)

    # Transpose -> (time, 40)
    mfcc = mfcc.T

    # Normalize
    mfcc = (mfcc - np.mean(mfcc)) / np.std(mfcc)

    # Pad or cut to 118 frames
    if mfcc.shape[0] < max_len:
        pad_width = max_len - mfcc.shape[0]
        mfcc = np.pad(mfcc, ((0, pad_width), (0, 0)), mode='constant')
    else:
        mfcc = mfcc[:max_len, :]

    # Reshape to (118, 1, 40)
    mfcc = mfcc.reshape(max_len, 1, 40)

    # Add batch and channel -> (1, 118, 1, 40)
    return np.expand_dims(mfcc, axis=0)




def convert_to_wav(path):
    ext = path.split(".")[-1].lower()
    if ext == "wav":
        return path
    new_path = path.replace(ext, "wav")
    audio = AudioSegment.from_file(path)
    audio.export(new_path, format="wav")
    return new_path

# ----------------------------
# 4) FLASK APP
# ----------------------------
app = Flask(__name__)

@app.route("/predict", methods=["POST"])
def predict():
    if "file" not in request.files:
        return jsonify({"error": "No audio file provided"}), 400

    f = request.files["file"]
    filepath = "/content/" + f.filename
    f.save(filepath)

    wav = convert_to_wav(filepath)
    features = extract_features(wav)

    preds = model.predict(features)
    idx = np.argmax(preds)
    confidence = float(np.max(preds))

    return jsonify({
        "predicted": labels[idx],
        "confidence": confidence
    })

@app.route("/")
def home():
    return "Audio classification API is running!"

# ----------------------------
# 5) START FLASK IN BACKGROUND
# ----------------------------
def run_app():
    app.run(port=5000, host="0.0.0.0")

thread = threading.Thread(target=run_app)
thread.start()

# ----------------------------
# 6) START NGROK
# ----------------------------
public_url = ngrok.connect(5000)
print("🔥 Your public API URL:", public_url)


 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.


🔥 Your public API URL: NgrokTunnel: "https://benign-georgene-connotive.ngrok-free.dev" -> "http://localhost:5000"


In [17]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 118, 1, 32)     │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 59, 1, 32)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 57, 1, 64)      │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 28, 1, 64)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1792)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       229,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 236,358 (923.28 KB)

 Trainable params: 236,356 (923.27 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

In [18]:
import requests

url = "https://benign-georgene-connotive.ngrok-free.dev/predict"
audio_path = "/content/drive/MyDrive/t1234.ogg"

with open(audio_path, "rb") as f:
    files = {"file": f}
    r = requests.post(url, files=files)

print(r.json())


ERROR:__main__:Exception on /predict [POST]
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/flask/app.py", line 1511, in wsgi_app
    response = self.full_dispatch_request()
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/flask/app.py", line 919, in full_dispatch_request
    rv = self.handle_user_exception(e)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/flask/app.py", line 917, in full_dispatch_request
    rv = self.dispatch_request()
         ^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/flask/app.py", line 902, in dispatch_request
    return self.ensure_sync(self.view_functions[rule.endpoint])(**view_args)  # type: ignore[no-any-return]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipython-input-786296188.py", line 66, in predict
    preds = model.predict(features)
            ^^^^^^^^^^^^^^^^^^^

JSONDecodeError: Expecting value: line 1 column 1 (char 0)